# Startup Profit Analysis
**Goal:** understand how R&D, Administration and Marketing spend affect startup profit, and build a Multiple Linear Regression model to predict profit.

Dataset: 50 startups (`profit_analysis_startup.csv`) with columns `RD_Spend`, `Administration`, `Marketing_Spend`, `State`, `Profit`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
df = pd.read_csv("profit_analysis_startup.csv")   # file is in the same folder as this notebook
df.head()

## 1. Data inspection & cleaning

In [ ]:
df.info()
print("Duplicates:", df.duplicated().sum())
print("Missing values:\n", df.isna().sum())
df.describe()

**Note:** some startups have `RD_Spend = 0` or `Marketing_Spend = 0`. These are valid values (no spend), not missing data, so they are kept.

In [ ]:
# Startups with zero spend - check how profitable they are
df[(df["RD_Spend"] == 0) | (df["Marketing_Spend"] == 0)]

## 2. Exploratory Data Analysis

In [ ]:
num_cols = ["RD_Spend", "Administration", "Marketing_Spend", "Profit"]
plt.figure(figsize=(6, 4.5))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="Blues")
plt.title("Correlation Heatmap")
plt.show()

R&D Spend is by far the strongest driver of profit (r ≈ 0.97). Marketing is moderately correlated (r ≈ 0.75) but also correlated with R&D (r ≈ 0.72). Administration is weakly related (r ≈ 0.20).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, col in zip(axes, num_cols[:3]):
    sns.regplot(data=df, x=col, y="Profit", ax=ax, scatter_kws={"alpha": 0.7}, line_kws={"color": "crimson"})
    ax.set_title(f"{col} vs Profit (r = {df[col].corr(df['Profit']):.2f})")
plt.tight_layout()
plt.show()

### State-wise analysis

In [ ]:
state_summary = df.groupby("State")["Profit"].agg(["count", "mean", "median", "sum"]).round(0)
state_summary

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x="State", y="Profit", estimator="mean", errorbar="sd", hue="State", palette="Blues_d", legend=False)
plt.title("Average Profit by State (error bars = 1 SD)")
plt.show()

Florida has the highest average profit, but the differences between states are small compared to the spread within each state (error bars overlap), so state is not a strong driver of profit.

## 3. Multiple Linear Regression

In [ ]:
features = ["RD_Spend", "Administration", "Marketing_Spend"]
X = df[features]
y = df["Profit"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"R2   : {r2_score(y_test, y_pred):.4f}")
print(f"MAE  : {mean_absolute_error(y_test, y_pred):,.0f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):,.0f}")
print()
print(pd.Series(model.coef_, index=features).round(4))
print("Intercept:", round(model.intercept_, 1))

**Interpreting the coefficients:** holding the other spends constant, each extra ₹/$1 spent on R&D is associated with about 0.80 extra profit. The Administration and Marketing coefficients are small, and Administration's is slightly negative.

### Is the 0.90 R² reliable? (only 10 test rows)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(LinearRegression(), X, y, cv=kf, scoring="r2")
print("5-fold R2 scores:", cv_r2.round(3))
print(f"Mean R2: {cv_r2.mean():.3f}  (std {cv_r2.std():.3f})")

# how much does R2 change with different random splits?
scores = []
for seed in range(100):
    a, b, c, d = train_test_split(X, y, test_size=0.2, random_state=seed)
    scores.append(r2_score(d, LinearRegression().fit(a, c).predict(b)))
print(f"100 random splits -> min {min(scores):.3f}, mean {np.mean(scores):.3f}, max {max(scores):.3f}")

With just 50 rows, the test R² depends on which rows land in the test set. Cross-validation gives a more trustworthy estimate (≈ 0.93).

### Multicollinearity check (VIF)

In [ ]:
def vif(frame):
    out = {}
    for col in frame.columns:
        others = frame.drop(columns=col)
        r2 = LinearRegression().fit(others, frame[col]).score(others, frame[col])
        out[col] = 1 / (1 - r2)
    return pd.Series(out).round(2)

vif(X)

All VIF values are well below 5, so multicollinearity is not a problem for this model.

### Does adding State help?

In [ ]:
X_state = pd.get_dummies(df[features + ["State"]], columns=["State"], drop_first=True)
cv_state = cross_val_score(LinearRegression(), X_state, y, cv=kf, scoring="r2")
print(f"5-fold R2 without State: {cv_r2.mean():.4f}")
print(f"5-fold R2 with State   : {cv_state.mean():.4f}")

Adding State (one-hot encoded) does not improve the model, so the final model uses the three spend features only.

### Residual analysis

In [ ]:
full_model = LinearRegression().fit(X, y)
pred_all = full_model.predict(X)
residuals = y - pred_all

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
axes[0].scatter(y_test, y_pred, alpha=0.8)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--")
axes[0].set(xlabel="Actual Profit", ylabel="Predicted Profit", title="Actual vs Predicted (test set)")
axes[1].scatter(pred_all, residuals, alpha=0.8)
axes[1].axhline(0, color="r", ls="--")
axes[1].set(xlabel="Predicted", ylabel="Residual", title="Residuals vs Predicted")
sns.histplot(residuals, kde=True, ax=axes[2])
axes[2].set_title("Residual Distribution")
plt.tight_layout()
plt.show()

df.assign(Predicted=pred_all.round(0), Residual=residuals.round(0)).loc[residuals.abs().nlargest(3).index]

Residuals are centred around 0 with no strong pattern. The biggest miss is the startup with **zero R&D spend** and the lowest profit (the model over-predicts it) - a low-R&D outlier that a straight line fits less well.

## 4. Predict profit for new startups

In [ ]:
new_startups = pd.DataFrame({
    "RD_Spend": [21892.92, 23940.93],
    "Administration": [81910.77, 96489.63],
    "Marketing_Spend": [164270.70, 137001.10],
})
new_startups["Predicted_Profit"] = full_model.predict(new_startups[features]).round(2)
new_startups

## 5. Conclusions
- **R&D Spend is the main driver of profit** (r = 0.97, coefficient ≈ 0.80).
- Marketing adds a little, Administration adds almost nothing.
- The model explains ~93% of profit variation (5-fold CV), with an average error of about 7,000 per startup.
- State does not meaningfully change profit once spending is accounted for.
- **Limitations:** only 50 rows, and the relationship is assumed linear. Predictions for startups far outside the observed spend ranges should not be trusted.